# LLM Embeddings — RAG on MPST Movie Synopses (Ollama)

**Embeddings** convert text into dense numeric vectors where similar texts land near each other in vector space.

This notebook:
1. Loads real movie synopses from the **MPST dataset** (~14 k films).
2. Generates embeddings via **Ollama** — fully local, no API key required.
3. Finds the most relevant synopsis for a natural-language query using **cosine similarity**.
4. Feeds the retrieved synopsis to a local chat model for a grounded answer — the core **Retrieval-Augmented Generation (RAG)** pattern.
5. Scales the whole pipeline with **LangChain + FAISS** and adds conversational memory.

In [ ]:
import ollama
import numpy as np
import pandas as pd
import textwrap

EMBED_MODEL = 'nomic-embed-text'   # ollama pull nomic-embed-text
CHAT_MODEL  = 'gemma3:4b'          # ollama pull gemma3:4b

available = [m.model for m in ollama.list().models]
print('Available Ollama models:', available)

missing = [m for m in [EMBED_MODEL, CHAT_MODEL] if not any(m in a for a in available)]
if missing:
    print('\n⚠  Pull missing models first:')
    for m in missing:
        print(f'  ollama pull {m}')
else:
    print('\n✓ Both required models are available')

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def top_docs(query_vec, doc_vecs, docs, k=3):
    scores = [cosine_similarity(query_vec, dv) for dv in doc_vecs]
    idx = np.argsort(scores)[::-1][:k]
    return [(docs[i], scores[i]) for i in idx]

---
# MPST Movie Synopses

The **Movie Plot Synopses with Tags** (MPST) dataset contains ~14 k IMDB film plot summaries paired with genre tags.  
Each synopsis becomes a *document* in our retrieval knowledge base.

In [ ]:
DATA_FOLDER = 'C:/Users/Graham West/Python Notebooks/Meharry Teaching/Datasets/'

df_mpst = pd.read_csv(DATA_FOLDER + 'MPST/mpst_full_data.csv')
print('Columns:', list(df_mpst.columns))
print(f'Shape  : {df_mpst.shape}')
df_mpst.head(3)

In [ ]:
df_mpst = df_mpst[['imdb_id', 'title', 'plot_synopsis', 'tags']].rename(columns={
    'imdb_id': 'tconst',
    'plot_synopsis': 'synopsis',
})
df_mpst['synopsis'] = df_mpst['synopsis'].str.strip()
df_mpst = df_mpst.dropna(subset=['synopsis']).reset_index(drop=True)

print(f'{len(df_mpst):,} movies with synopses')
print(f'Avg synopsis length: {df_mpst.synopsis.str.len().mean():.0f} chars')
df_mpst[['title', 'tags', 'synopsis']].head(3)

---
# Part 1 — Manual RAG with Direct Ollama

No frameworks. We call `ollama.embed()` for vectors and `ollama.chat()` for answers.  
Retrieval is plain cosine similarity on numpy arrays.

We sample **200 movies** so the embedding loop completes quickly in a demo.

In [ ]:
N_SAMPLE = 200
MAX_CHARS = 2000   # cap synopsis length; nomic-embed-text handles up to ~8 k tokens

df_sample = df_mpst.sample(N_SAMPLE, random_state=42).reset_index(drop=True)
doc_texts  = df_sample['synopsis'].str[:MAX_CHARS].tolist()
doc_titles = df_sample['title'].tolist()

print(f'Embedding {N_SAMPLE} synopses with "{EMBED_MODEL}"...')
doc_vecs = []
for i, text in enumerate(doc_texts):
    vec = ollama.embed(model=EMBED_MODEL, input=text)['embeddings'][0]
    doc_vecs.append(vec)
    if (i + 1) % 50 == 0:
        print(f'  {i + 1}/{N_SAMPLE} done')

print(f'\nDone — each vector has {len(doc_vecs[0])} dimensions')

In [ ]:
QUERY = 'A heist movie where criminals plan and execute a robbery'

query_vec = ollama.embed(model=EMBED_MODEL, input=QUERY)['embeddings'][0]

results = top_docs(query_vec, doc_vecs, list(zip(doc_titles, doc_texts)), k=3)

print(f'Query: {QUERY}\n')
for rank, ((title, synopsis), score) in enumerate(results, 1):
    print(f'Rank {rank}  (score={score:.3f}): {title}')
    print(textwrap.fill(synopsis[:300] + ' ...', width=80))
    print()

In [ ]:
best_title, best_synopsis = results[0][0]

rag_prompt = (
    f'Context — synopsis of "{best_title}":\n{best_synopsis[:1200]}\n\n'
    f'Question: {QUERY}\n\n'
    f'Using only the context above, explain why this movie fits the question.'
)

ans = ollama.chat(
    model=CHAT_MODEL,
    messages=[{'role': 'user', 'content': rag_prompt}]
)
print(f'RAG Answer:\n{ans["message"]["content"]}')

---
# Part 2 — LangChain RAG Pipeline

LangChain adds composable chains, a unified retriever interface, and text chunking so long synopses don't overwhelm the embedding context window.

We index **500 MPST synopses** into a FAISS vector store and build a full conversational RAG chain.

| Capability | Direct `ollama` | LangChain |
|---|---|---|
| Text chunking | manual slicing | `RecursiveCharacterTextSplitter` |
| Vector index | manual cosine loops | `FAISS.as_retriever()` |
| Prompt wiring | f-strings | `ChatPromptTemplate` + LCEL `\|` pipes |
| Conversational memory | manual list | explicit `chat_history` + contextualize step |

In [ ]:
import time

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage, AIMessage

llm        = ChatOllama(model=CHAT_MODEL, temperature=0.2)
embeddings = OllamaEmbeddings(model=EMBED_MODEL)

print(f'LLM      : {CHAT_MODEL}')
print(f'Embedder : {EMBED_MODEL}')

# Warm up the LLM so the first real call is fast
_ = llm.invoke('hi')
print('LLM ready')

In [ ]:
LC_SAMPLE = 500   # increase for broader coverage

df_lc = df_mpst.sample(LC_SAMPLE, random_state=0).reset_index(drop=True)

lc_docs = [
    Document(
        page_content=row['synopsis'][:3000],
        metadata={'title': row['title'], 'tconst': row['tconst'], 'tags': row['tags']}
    )
    for _, row in df_lc.iterrows()
]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=80,
    separators=['\n\n', '\n', '. ', ' ', ''],
)
chunks = splitter.split_documents(lc_docs)
print(f'Split {len(lc_docs)} synopses → {len(chunks)} chunks')

print(f'\nEmbedding {len(chunks)} chunks with "{EMBED_MODEL}"...')
t0 = time.perf_counter()
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f'FAISS index built in {time.perf_counter() - t0:.1f}s  ({vectorstore.index.ntotal} vectors)')

## The Retriever

A **retriever** wraps the vector store and handles semantic search.  
`k=3` means: return the 3 chunks whose embeddings are closest to the query embedding.  
You can call `.invoke()` directly to inspect what the retriever finds — useful for debugging why a RAG answer is good or bad.

In [ ]:
retriever = vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 3})

test_query = 'A thriller where a detective hunts a serial killer'
retrieved  = retriever.invoke(test_query)

print(f'Query: "{test_query}"\n')
for i, doc in enumerate(retrieved, 1):
    title = doc.metadata.get('title', 'Unknown')
    print(f'--- Chunk {i}: {title} ---')
    print(textwrap.fill(doc.page_content[:300] + ' ...', width=80))
    print()

## The RAG Chain (LCEL)

LangChain Expression Language (LCEL) uses `|` to chain components:

```
user question
     ↓
 retriever  →  fetches top-k relevant synopsis chunks
     ↓
 prompt template  →  inserts chunks as "context" + original question
     ↓
 LLM  →  generates answer grounded in retrieved context
     ↓
 string parser  →  returns clean text
```

In [ ]:
RAG_PROMPT = ChatPromptTemplate.from_messages([
    ('system',
     'You are a helpful movie expert. Answer using ONLY the provided movie synopses. '
     'If none of the synopses match the question, say so. '
     'Always name the movie and explain why it fits.\n\n'
     'Context (retrieved synopses):\n{context}'),
    ('human', '{question}'),
])

def format_docs(docs):
    return '\n\n---\n\n'.join(
        f"[{doc.metadata.get('title', '?')}]\n{doc.page_content.strip()}"
        for doc in docs
    )

rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print('RAG chain assembled:')
print('  OllamaEmbeddings  →  FAISS similarity search')
print('  ChatPromptTemplate →  injects retrieved synopses as context')
print(f'  ChatOllama         →  {CHAT_MODEL}')

In [ ]:
questions = [
    'What sci-fi movie involves artificial intelligence turning against humans?',
    'Recommend a movie where the protagonist seeks revenge for a family member.',
    'Find a horror movie set in an isolated location with a small group of survivors.',
]

for q in questions:
    print(f'Q: {q}')
    t0 = time.perf_counter()
    answer = rag_chain.invoke(q)
    print(f'A: {textwrap.fill(answer, width=75)}')
    print(f'   [{time.perf_counter() - t0:.1f}s]\n')

---
## Conversational RAG with Memory

Multi-turn conversation where follow-up questions can reference earlier answers.

A **contextualization** step rewrites vague follow-ups into standalone questions
before they hit the retriever — otherwise *"Is there a sequel?"* embeds poorly
and retrieves irrelevant chunks.

Chat history is a plain Python list of `HumanMessage` / `AIMessage` objects.
No memory objects or session stores needed.

In [ ]:
contextualize_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'Given the chat history and the latest user question about movies, rewrite it '
     'as a fully self-contained standalone question. Return ONLY the rewritten question.'),
    MessagesPlaceholder('chat_history'),
    ('human', '{input}'),
])
contextualize_chain = contextualize_prompt | llm | StrOutputParser()

qa_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a helpful movie expert. Answer using ONLY the provided synopses. '
     'Name the movie and explain why it fits.\n\nContext:\n{context}'),
    MessagesPlaceholder('chat_history'),
    ('human', '{input}'),
])

chat_history = []

def ask(question: str) -> str:
    t0 = time.perf_counter()
    standalone = (
        contextualize_chain.invoke({'input': question, 'chat_history': chat_history})
        if chat_history else question
    )
    docs     = retriever.invoke(standalone)
    context  = format_docs(docs)
    messages = qa_prompt.format_messages(
        context=context, chat_history=chat_history, input=question
    )
    answer = llm.invoke(messages).content
    chat_history.extend([HumanMessage(content=question), AIMessage(content=answer)])
    sources = list({d.metadata.get('title', '?') for d in docs})
    print(f'Q: {question}')
    print(f'A: {textwrap.fill(answer, width=75)}')
    print(f'   [retrieved: {", ".join(sources[:3])} | {time.perf_counter() - t0:.1f}s]\n')
    return answer

In [ ]:
chat_history.clear()

_ = ask('Find me a movie about a soldier dealing with the trauma of war after coming home.')
_ = ask('Are there any similar movies in the database but with a more uplifting ending?')
_ = ask('Which of those would you recommend for someone who also likes psychological dramas?')

In [ ]:
print('Conversation history:')
print('-' * 55)
for msg in chat_history:
    role = 'Human' if msg.type == 'human' else 'AI'
    print(f'[{role}]\n{textwrap.fill(msg.content, width=75)}\n')

---
## Try Your Own Question

Ask anything and see what MPST synopses the retriever surfaces.
Answers are grounded in whatever the vector store finds — if the database doesn't have a match, the LLM will say so.

To start a fresh conversation:
```python
chat_history.clear()
```

Ideas:
- *"What movie has a twist ending where the narrator is revealed to be the villain?"*
- *"Find a coming-of-age film set in the 1980s."*
- *"Which movie involves a character who discovers they have superpowers?"*

In [ ]:
chat_history.clear()

my_question = 'What movie features a character who travels through time to prevent a disaster?'
_ = ask(my_question)